In [ ]:
import psycopg2
import nltk
import numpy as np
from nltk.stem.snowball import FrenchStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt')
nltk.download('punkt_tab')

conn = psycopg2.connect(
    host="localhost",
    database="ecommerce_rec",
    user="postgres",
    password="votre_mot_de_passe",
    port="5432"
)
cursor = conn.cursor()

# RÉCUPÉRER PRODUITS
cursor.execute("SELECT ID, NomPdt, Description FROM Produit")
produits = cursor.fetchall()
print(f"✅ {len(produits)} produits récupérés")

# NLP
stopwords_fr = {'le','la','les','un','une','des','et','est','en',
                'de','du','au','avec','pour','que','qui','se','sur'}
stemmer = FrenchStemmer()
dict_produits = {}

for p in produits:
    id_p, nom, desc = p
    mots = nltk.word_tokenize(desc.lower())
    stems = [stemmer.stem(m) for m in mots
             if m.isalpha() and m not in stopwords_fr]
    dict_produits[id_p] = " ".join(stems)

print("✅ NLP terminé")

# VECTEUR BINAIRE
ids = sorted(dict_produits.keys())
vocab = sorted(set(m for s in dict_produits.values() for m in s.split()))
vocab_index = {m: i for i, m in enumerate(vocab)}

def vecteur_binaire(stems_str):
    v = np.zeros(len(vocab))
    for m in stems_str.split():
        if m in vocab_index:
            v[vocab_index[m]] = 1
    return v

matrice_binaire = np.array([vecteur_binaire(dict_produits[i]) for i in ids])
print(f"✅ Vecteurs binaires : {matrice_binaire.shape}")

# TF-IDF
corpus = [dict_produits[i] for i in ids]
tfidf = TfidfVectorizer()
matrice_tfidf = tfidf.fit_transform(corpus).toarray()
print("✅ Vecteurs TF-IDF calculés")

# SIMILARITÉ COSINUS
sim_cosinus = cosine_similarity(matrice_tfidf)
print("✅ Matrice similarité Cosinus:")
print(np.round(sim_cosinus, 2))

# SIMILARITÉ JACCARD
def jaccard(v1, v2):
    intersection = np.sum(np.minimum(v1, v2))
    union = np.sum(np.maximum(v1, v2))
    return intersection / union if union > 0 else 0

sim_jaccard = np.array([[jaccard(matrice_binaire[i], matrice_binaire[j])
                         for j in range(len(ids))] for i in range(len(ids))])
print("✅ Matrice similarité Jaccard calculée")

# SAUVEGARDER DANS BD
cursor.execute("DELETE FROM SimilariteContenu")
for i, id_i in enumerate(ids):
    for j, id_j in enumerate(ids):
        if i != j:
            score = float(sim_cosinus[i][j])
            cursor.execute("""
                INSERT INTO SimilariteContenu (IDProduit1, IDProduit2, Score)
                VALUES (%s, %s, %s)
                ON CONFLICT (IDProduit1, IDProduit2) DO UPDATE SET Score=%s
            """, (id_i, id_j, score, score))
conn.commit()
print("✅ Similarités sauvegardées dans BD!")

# TOP-N PAR CONTENU
def recommander_contenu(id_produit, N=3):
    cursor.execute("""
        SELECT p.ID, p.NomPdt, p.Categorie, p.Prix, s.Score
        FROM SimilariteContenu s
        JOIN Produit p ON s.IDProduit2 = p.ID
        WHERE s.IDProduit1 = %s
        ORDER BY s.Score DESC LIMIT %s
    """, (id_produit, N))
    resultats = cursor.fetchall()
    print(f"\n🛒 Top {N} similaires au produit ID {id_produit}:")
    for r in resultats:
        print(f"  → {r[1]} ({r[2]}) | {r[3]}€ | Similarité: {r[4]:.2f}")
        print(f"     💡 Recommandé car similaire à votre produit préféré")
    return resultats

recommander_contenu(1, 3)
recommander_contenu(5, 3)
recommander_contenu(13, 3)

conn.close()
print("\n✅ Filtrage contenu terminé!")